# DeepGuard — DF40 one-click downloader

Run the single code cell below. It mounts Google Drive, resolves the official DF40 testing folder, tests the first file, and then downloads all files directly to Drive. It records progress so the run can be resumed.


In [ ]:
import subprocess,sys,json,shutil,time
from pathlib import Path

# Install a current gdown
subprocess.run([sys.executable,'-m','pip','install','-q','-U','gdown>=6.1.0'],check=True)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

ROOT=Path('/content/drive/MyDrive/DeepGuard')
DEST=ROOT/'datasets/DF40'
MAN=ROOT/'manifests'
DEST.mkdir(parents=True,exist_ok=True); MAN.mkdir(parents=True,exist_ok=True)
URL='https://drive.google.com/drive/folders/1U8meBbqVvmUkc5GD0jxct6xe6Gwk9wKD?usp=drive_link'

print('DF40 destination:',DEST)
print('Available Drive space (GiB):',round(shutil.disk_usage('/content/drive').free/1024**3,1))

# Resolve the official public folder.
print('\n[1/3] Reading official DF40 folder...')
r=subprocess.run([sys.executable,'-m','gdown','--folder',URL,'--json'],capture_output=True,text=True)
raw=r.stdout.strip(); start=raw.find('[')
if r.returncode!=0 or start<0:
    print(r.stdout[-3000:]); print(r.stderr[-6000:])
    raise RuntimeError('Could not read the official DF40 folder.')
items=json.loads(raw[start:])
(MAN/'df40_gdrive_file_manifest.json').write_text(json.dumps(items,indent=2))
print('Files found:',len(items))

# Test the first actual file before starting the large download.
print('\n[2/3] Testing one DF40 file...')
test=None
for x in items:
    if x.get('url') and x.get('path'):
        test=x; break
if test is None: raise RuntimeError('No downloadable file was found in the official folder.')
print('Test:',test['path'])
test_path=Path('/content/df40_download_test.tmp')
rr=subprocess.run([sys.executable,'-m','gdown',test['url'],'-O',str(test_path)],capture_output=True,text=True)
if rr.returncode!=0 or not test_path.exists() or test_path.stat().st_size==0:
    print('--- gdown stdout ---'); print(rr.stdout[-4000:])
    print('--- gdown stderr ---'); print(rr.stderr[-8000:])
    raise RuntimeError('The official DF40 file could not be downloaded. The diagnostic above shows why.')
print('Test download OK:',round(test_path.stat().st_size/1024**2,2),'MiB')
test_path.unlink(missing_ok=True)

# Download each file directly to Drive.
print('\n[3/3] Downloading DF40 to Drive...')
progress={'total':len(items),'completed':0,'skipped':0,'failed':[]}
for i,x in enumerate(items,1):
    rel=x.get('path','').lstrip('/')
    url=x.get('url')
    if not rel or not url:
        progress['failed'].append([rel,'missing path/url']); continue
    target=DEST/rel; target.parent.mkdir(parents=True,exist_ok=True)
    if target.exists() and target.stat().st_size>0:
        progress['skipped']+=1; continue
    print(f'[{i}/{len(items)}] {rel}',flush=True)
    rr=subprocess.run([sys.executable,'-m','gdown',url,'-O',str(target),'--continue'],capture_output=True,text=True)
    if rr.returncode==0 and target.exists() and target.stat().st_size>0:
        progress['completed']+=1
    else:
        progress['failed'].append([rel,'exit='+str(rr.returncode),rr.stderr[-1000:]])
    (MAN/'df40_download_progress.json').write_text(json.dumps(progress,indent=2))

files=[p for p in DEST.rglob('*') if p.is_file()]
size=sum(p.stat().st_size for p in files)
print('\n==============================')
print('DF40 DOWNLOAD REPORT')
print('==============================')
print('Files found in Drive:',len(files))
print('Size in Drive (GiB):',round(size/1024**3,2))
print('New downloads:',progress['completed'])
print('Already present:',progress['skipped'])
print('Failed:',len(progress['failed']))
print('Destination:',DEST)
if progress['failed']:
    print('First failure:',progress['failed'][0])
    raise RuntimeError('Some DF40 files failed. The report above identifies the first one; rerun to resume.')
print('STATUS: COMPLETE')
